# Research 07 — Fast OOF combination screen for one-vs-all models

The one-vs-all models individually underperformed, but LightGBM OVA showed a tiny gain in a fixed 80/20 blend. OVA class scores do not sum to one per row, so averaging them directly changes their effective weight row by row. **Hypotheses:** normalizing OVA scores before blending may make the mixture more coherent, and a small fixed OVA contribution may complement other OOF predictors. We test predeclared 80/10/10 combinations and one confidence-margin selector; this is not a parameter sweep. All rows are training OOF predictions; no Test data is loaded.

Repeated reuse of these folds makes all changes screening-only. Keep the established candidate unless a gain is convincing and stable across folds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"])
y = train.segment.to_numpy(dtype=int)
base_frame = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base_frame) == len(y) and np.array_equal(base_frame.segment.to_numpy(), y)


def load_probabilities(filename, normalize=False):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and np.all(probs >= 0), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert np.all(row_sums > 0), filename
    if normalize:
        if not np.allclose(row_sums, 1, atol=1e-5):
            print(
                f"{filename}: normalizing row sums in [{row_sums.min():.3f}, {row_sums.max():.3f}]"
            )
        probs = probs / row_sums
        assert np.allclose(probs.sum(axis=1), 1, atol=1e-5), filename
    return probs


p_base = load_probabilities("segment-oof.csv", normalize=True)
p_xgb = load_probabilities("segment-xgboost-oof.csv", normalize=True)
p_balanced = load_probabilities("segment-balanced-catboost-oof.csv", normalize=True)
p_catova = load_probabilities("segment-catboost-ova-oof.csv", normalize=True)
p_lgbova_raw = load_probabilities("segment-lightgbm-ova-oof.csv")
p_lgbova = load_probabilities("segment-lightgbm-ova-oof.csv", normalize=True)
print(f"Validated five aligned OOF probability sets for {len(y):,} training rows.")

segment-catboost-ova-oof.csv: normalizing row sums in [0.594, 1.653]
segment-lightgbm-ova-oof.csv: normalizing row sums in [0.271, 1.764]
Validated five aligned OOF probability sets for 104,000 training rows.


## Predeclared alternatives

Compare the existing selected blend, its earlier fixed 80/10/10 mixture, raw versus normalized 80/20 LightGBM OVA blends, and three fixed 80/10/10 mixtures that add normalized LightGBM OVA alongside one other model. The final candidate uses a label-free rule: choose the base or normalized LightGBM OVA probabilities row-wise according to which has the larger top-two probability margin. No weights or thresholds are fitted to labels.

In [2]:
candidates = {
    "Current CatBoost/LightGBM": p_base,
    "Existing 80/10/10 (XGB + balanced CatBoost)": 0.8 * p_base + 0.1 * p_xgb + 0.1 * p_balanced,
    "80/20 current + raw LightGBM OVA (reproduction)": 0.8 * p_base + 0.2 * p_lgbova_raw,
    "80/20 current + normalized LightGBM OVA": 0.8 * p_base + 0.2 * p_lgbova,
    "80/10/10 current + LightGBM OVA + XGBoost": 0.8 * p_base + 0.1 * p_lgbova + 0.1 * p_xgb,
    "80/10/10 current + LightGBM OVA + balanced CatBoost": 0.8 * p_base
    + 0.1 * p_lgbova
    + 0.1 * p_balanced,
    "80/10/10 current + LightGBM OVA + CatBoost OVA": 0.8 * p_base
    + 0.1 * p_lgbova
    + 0.1 * p_catova,
}


def top_margin(probs):
    ordered = np.sort(probs, axis=1)
    return ordered[:, -1] - ordered[:, -2]


use_ova = top_margin(p_lgbova) > top_margin(p_base)
candidates["Margin selector: base vs LightGBM OVA"] = np.where(use_ova[:, None], p_lgbova, p_base)
print(f"Margin selector chooses LightGBM OVA on {use_ova.mean():.1%} of rows.")


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
folds = list(cv.split(np.zeros(len(y)), y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)

results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current CatBoost/LightGBM",
    [f"fold_{fold}_macro_f1" for fold in range(1, 4)],
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.sort_values("macro_f1", ascending=False).round(6))
results.to_csv(ROOT / "assets/segment-ova-combination-screen.csv", index=False)


def complementarity(name, probs):
    base_correct = p_base.argmax(axis=1) == y
    model_correct = probs.argmax(axis=1) == y
    return {
        "model": name,
        "base_errors_fixed": int((~base_correct & model_correct).sum()),
        "base_correct_harmed": int((base_correct & ~model_correct).sum()),
    }


complement = pd.DataFrame(
    [
        complementarity("LightGBM OVA", p_lgbova),
        complementarity("CatBoost OVA", p_catova),
        complementarity("XGBoost", p_xgb),
        complementarity("Balanced CatBoost", p_balanced),
    ]
)
display(complement)
complement.to_csv(ROOT / "assets/segment-ova-error-complementarity.csv", index=False)
best = results.sort_values("macro_f1", ascending=False).iloc[0]
baseline = results.loc[results.candidate == "Current CatBoost/LightGBM", "macro_f1"].iloc[0]
print(f"Best screen: {best.candidate}; macro delta={best.macro_f1 - baseline:+.6f}.")
print(
    "OOF reuse makes this screening evidence only; no Test predictions or submission were generated."
)

Margin selector chooses LightGBM OVA on 30.6% of rows.


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
1,Existing 80/10/10 (XGB + balanced CatBoost),0.751142,0.764884,0.749915,0.750579,0.752899,0.000470,0.000758,-0.000491
5,80/10/10 current + LightGBM OVA + balanced Cat...,0.751026,0.764746,0.749683,0.750025,0.753338,0.000238,0.000204,-0.000051
2,80/20 current + raw LightGBM OVA (reproduction),0.750985,0.764753,0.748985,0.750392,0.753545,-0.000460,0.000571,0.000155
3,80/20 current + normalized LightGBM OVA,0.750957,0.764749,0.748877,0.750497,0.753464,-0.000568,0.000676,0.000074
0,Current CatBoost/LightGBM,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
4,80/10/10 current + LightGBM OVA + XGBoost,0.750813,0.764706,0.748984,0.750165,0.753258,-0.000461,0.000344,-0.000132
6,80/10/10 current + LightGBM OVA + CatBoost OVA,0.750571,0.764475,0.749063,0.749438,0.753178,-0.000382,-0.000383,-0.000211
7,Margin selector: base vs LightGBM OVA,0.749668,0.763700,0.747104,0.749126,0.752741,-0.002341,-0.000695,-0.000649


,model,base_errors_fixed,base_correct_harmed
0,LightGBM OVA,1784,2136
1,CatBoost OVA,1976,2563
2,XGBoost,2082,2474
3,Balanced CatBoost,3730,5235


Best screen: Existing 80/10/10 (XGB + balanced CatBoost); macro delta=+0.000247.
OOF reuse makes this screening evidence only; no Test predictions or submission were generated.


## Decision

These combinations only test whether already-trained OOF predictors add useful signal; no model is retrained. Compare overall macro and weighted F1 with the fold deltas before drawing conclusions. Small OOF gains are vulnerable to selection noise, and repeated experiments on the same folds are not an independent validation. Test rows and `submit/index.csv` remain untouched.